# EDA — Job Satisfaction Among Software Developers

**Pipeline overview**
1. Load & target filter
2. Train / test split *(all data-dependent ops happen inside the train fold)*
3. Imputation (train-derived statistics only)
4. Feature relevance — Cramér's V for categoricals, Kruskal-Wallis for numerics
5. Ordinal inter-correlation (Spearman — survey items only)
6. Salary visualisation
7. Random Forest feature importance
8. Champion vs Challenger (raw survey cols vs aggregated themes)
9. Currency analysis — raw vs `is_foreign_currency`


## 1. Load & target filter

In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as ss
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

df_raw = pd.read_csv("train_data.csv", low_memory=False)

# Only row-drop allowed globally: rows where the target is missing.
df_raw = df_raw.dropna(subset=["JobSatLevel"]).reset_index(drop=True)

TARGET = "JobSatLevel"

ENDORSE_COLS = sorted([c for c in df_raw.columns
                       if c.startswith("TechEndorse_") and c != "TechEndorseIntro"
                       and "_TEXT" not in c])
OPPOSE_COLS  = sorted([c for c in df_raw.columns
                       if c.startswith("TechOppose_")
                       and "_TEXT" not in c])
SURVEY_COLS  = ENDORSE_COLS + OPPOSE_COLS

print(f"Rows after target filter : {len(df_raw):,}")
print(f"Endorse survey cols ({len(ENDORSE_COLS)}): {ENDORSE_COLS}")
print(f"Oppose  survey cols ({len(OPPOSE_COLS)}): {OPPOSE_COLS}")
print(f"\nTarget distribution:\n{df_raw[TARGET].value_counts().sort_index()}")


## 2. Currency → USD conversion

Exchange rates are external constants — no statistics are learned from data here,
so this step is safe to apply before the split.


In [ ]:
RATES_TO_USD = {
    'USD': 1.0, 'EUR': 1.08, 'GBP': 1.25, 'INR': 0.012, 'CAD': 0.74,
    'AUD': 0.65, 'BRL': 0.20, 'PLN': 0.25, 'SEK': 0.095, 'CHF': 1.10,
    'JPY': 0.0066, 'CZK': 0.043, 'HUF': 0.0028, 'MYR': 0.21, 'EGP': 0.021,
    'LKR': 0.0033, 'RSD': 0.0092, 'RON': 0.22, 'UYU': 0.026, 'AED': 0.27,
    'ARS': 0.0012, 'RUB': 0.011, 'NOK': 0.092, 'CRC': 0.002, 'PHP': 0.018,
    'ILS': 0.27,  'BGN': 0.55, 'MAD': 0.10, 'MXN': 0.06, 'TRY': 0.031,
    'BOB': 0.14,  'NPR': 0.0075, 'ZAR': 0.053, 'TND': 0.32, 'CNY': 0.14,
    'PKR': 0.0036, 'SGD': 0.74, 'AZN': 0.59, 'DKK': 0.14, 'NGN': 0.0007,
    'IRR': 0.000024, 'HKD': 0.13, 'TWD': 0.031, 'VND': 0.00004, 'CLP': 0.0011,
    'KRW': 0.00074, 'COP': 0.00026, 'UGX': 0.00026, 'JOD': 1.41, 'IDR': 0.000063,
    'UAH': 0.025, 'BDT': 0.009, 'ANG': 0.55, 'DOP': 0.017, 'GTQ': 0.13,
    'QAR': 0.27, 'BAM': 0.55, 'AMD': 0.0025, 'THB': 0.027, 'KZT': 0.0022,
    'GEL': 0.37, 'GHS': 0.075, 'KES': 0.0075, 'NZD': 0.60, 'PEN': 0.27,
    'BHD': 2.65, 'SAR': 0.27, 'OMR': 2.60, 'KWD': 3.25, 'NON': 1.0,
}

df_raw['Clean_Currency'] = df_raw['Currency'].astype(str).str[:3].str.upper()
df_raw['ExchangeRate']   = df_raw['Clean_Currency'].map(RATES_TO_USD).fillna(1.0)
df_raw['CompTotal_USD']  = df_raw['CompTotal'] * df_raw['ExchangeRate']

missing_codes = (
    df_raw.loc[df_raw['ExchangeRate'] == 1.0, 'Clean_Currency']
    .loc[df_raw['Clean_Currency'] != 'USD']
    .unique()
)
print(f"CompTotal_USD computed. Non-null: {df_raw['CompTotal_USD'].notna().sum():,}")
if len(missing_codes):
    print(f"Unmapped codes (defaulted to 1.0): {missing_codes}")


## 3. Train / test split

**Everything data-dependent from here on is fit on `df_train` only.**


In [ ]:
from sklearn.model_selection import train_test_split

df_train, df_test = train_test_split(
    df_raw, test_size=0.2, random_state=42, stratify=df_raw[TARGET]
)
df_train = df_train.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)

print(f"Train: {len(df_train):,}  |  Test: {len(df_test):,}")
print(f"Target distribution (train):\n{df_train[TARGET].value_counts().sort_index()}")


## 4. Imputation (fit on train fold only)

| Feature group | Strategy | Rationale |
|---|---|---|
| `Country` | Mode from train | Low missingness, nominal |
| `Currency` | Country→Currency mode map from train; fallback `USD` | Avoids unseen-country null propagation in test |
| `CompTotal_USD` | Left as-is (handled by imputer in modelling cells) | Median imputation would use test data if done globally |
| Survey ordinals | Sentinel `−1` | Outside the 1–5 scale; signals "question skipped"; tree models split on it cleanly |
| `TechEndorseIntro` | `'Skipped'` string | Nominal category for "no response" |


In [ ]:
# --- Country ---
country_mode_train = df_train['Country'].mode()[0]
df_train['Country'] = df_train['Country'].fillna(country_mode_train)
df_test['Country']  = df_test['Country'].fillna(country_mode_train)

# --- Currency (inferred from Country using train map) ---
country_to_currency_train = (
    df_train.dropna(subset=['Country', 'Currency'])
    .groupby('Country')['Currency']
    .agg(lambda x: x.mode()[0])
    .to_dict()
)
df_train['Currency'] = df_train['Currency'].fillna(df_train['Country'].map(country_to_currency_train)).fillna('USD')
df_test['Currency']  = df_test['Currency'].fillna(df_test['Country'].map(country_to_currency_train)).fillna('USD')

# --- TechEndorseIntro ---
df_train['TechEndorseIntro'] = df_train['TechEndorseIntro'].fillna('Skipped')
df_test['TechEndorseIntro']  = df_test['TechEndorseIntro'].fillna('Skipped')

# --- Survey ordinals: sentinel -1 for skipped questions ---
df_train[SURVEY_COLS] = df_train[SURVEY_COLS].fillna(-1)
df_test[SURVEY_COLS]  = df_test[SURVEY_COLS].fillna(-1)

print("Post-imputation nulls (train):")
check_cols = ['Country', 'Currency', 'TechEndorseIntro'] + SURVEY_COLS
print(df_train[check_cols].isnull().sum()[lambda s: s > 0].to_string() or "  None")
print("\nPost-imputation nulls (test):")
print(df_test[check_cols].isnull().sum()[lambda s: s > 0].to_string() or "  None")


## 5. Salary outlier cap

The 99th-percentile threshold is derived from the **training fold only** and applied
as a **clip (winsorise)** to both folds — no rows are dropped.


In [ ]:
q99_train = df_train['CompTotal_USD'].quantile(0.99)
df_train['CompTotal_USD'] = df_train['CompTotal_USD'].clip(upper=q99_train)
df_test['CompTotal_USD']  = df_test['CompTotal_USD'].clip(upper=q99_train)

print(f"99th-pct cap (from train): ${q99_train:,.0f} USD")
print(f"Train CompTotal_USD range: {df_train['CompTotal_USD'].min():,.0f} – {df_train['CompTotal_USD'].max():,.0f}")


## 6. Feature relevance (train fold only)

| Feature type | Method | Why |
|---|---|---|
| Categorical → Categorical | Cramér's V + χ² p-value | Measures association strength, corrected for table size |
| Ordinal/Numerical → Categorical target | Kruskal-Wallis H | Non-parametric; no normality assumption |

**Spearman correlation is only run on ordinal survey items** (numeric scale 1–5 + sentinel −1),
never on nominal categoricals like Country or Currency.


In [ ]:
def cramers_v(x, y):
    ct   = pd.crosstab(x, y)
    chi2 = ss.chi2_contingency(ct)[0]
    n    = ct.sum().sum()
    r, k = ct.shape
    phi2corr = max(0, chi2 / n - ((k - 1) * (r - 1)) / (n - 1))
    rcorr = r - ((r - 1) ** 2) / (n - 1)
    kcorr = k - ((k - 1) ** 2) / (n - 1)
    m = min(kcorr - 1, rcorr - 1)
    return 0.0 if m <= 0 else np.sqrt(phi2corr / m)

# Categorical features (nominal): Cramér's V only
CAT_FEATURES = ['Currency', 'Country', 'TechEndorseIntro']

# Ordinal / numeric features: Kruskal-Wallis only
ORD_NUM_FEATURES = ['CompTotal_USD'] + SURVEY_COLS

cat_results, kw_results = [], []

for col in CAT_FEATURES:
    temp = df_train[[col, TARGET]].dropna()
    if temp.empty:
        continue
    cv = cramers_v(temp[col].astype(str), temp[TARGET])
    _, p, _, _ = ss.chi2_contingency(pd.crosstab(temp[col].astype(str), temp[TARGET]))
    cat_results.append({'Feature': col, 'Method': "Cramér's V", 'Score': round(cv, 4), 'p-value': f'{p:.4e}'})

for col in ORD_NUM_FEATURES:
    temp = df_train[[col, TARGET]].dropna()
    if temp.empty:
        continue
    groups = [g[col].values for _, g in temp.groupby(TARGET)]
    if len(groups) < 2:
        continue
    stat, p = ss.kruskal(*groups)
    kw_results.append({'Feature': col, 'Method': 'Kruskal-Wallis H', 'Score': round(stat, 4), 'p-value': f'{p:.4e}'})

cat_df = pd.DataFrame(cat_results).sort_values('Score', ascending=False)
kw_df  = pd.DataFrame(kw_results).sort_values('Score', ascending=False)

print("=== Categorical features — Cramér's V ===")
print(cat_df.to_string(index=False))
print("\n=== Ordinal / Numerical features — Kruskal-Wallis H ===")
print(kw_df.to_string(index=False))


## 7. Survey-item inter-correlation (Spearman)

Spearman is appropriate here because survey items are **ordinal** (1–5 scale + −1 sentinel).
It is **not** applied to nominal categoricals (Country, Currency, TechEndorseIntro).

Pairs with |r| ≥ 0.7 are flagged as redundant candidates for grouping or dropping.


In [ ]:
# Replace sentinel with NaN so -1 doesn't distort the rank correlation
survey_for_corr = df_train[SURVEY_COLS].replace(-1, np.nan)

corr_matrix = survey_for_corr.corr(method='spearman')
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

high_pairs = [
    {'Feature A': r, 'Feature B': c, 'Spearman r': round(upper.loc[r, c], 4)}
    for c in upper.columns
    for r in upper.index
    if pd.notna(upper.loc[r, c]) and abs(upper.loc[r, c]) >= 0.7
]

if high_pairs:
    corr_pairs_df = pd.DataFrame(high_pairs).sort_values('Spearman r', key=abs, ascending=False)
    print("--- Highly correlated pairs (|r| ≥ 0.7) ---")
    print(corr_pairs_df.to_string(index=False))
    print("\nRecommendation: drop one of each pair, or aggregate into a theme score.")
else:
    print("No highly correlated pairs found (|r| ≥ 0.7). Survey items are relatively independent.")


## 8. Salary distribution by satisfaction level (train)

In [ ]:
sns.set_theme(style="whitegrid")

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Box plot
sns.boxplot(data=df_train, x=TARGET, y='CompTotal_USD', ax=axes[0], palette='viridis')
axes[0].set_title('Salary (USD, winsorised at 99th pct)', fontsize=13)
axes[0].set_xlabel('Job Satisfaction Level')
axes[0].set_ylabel('Total Compensation (USD)')
axes[0].tick_params(axis='x', rotation=30)

# Violin
sns.violinplot(data=df_train, x=TARGET, y='CompTotal_USD', ax=axes[1],
               palette='mako', inner='quartile')
axes[1].set_title('Salary Distribution — Violin', fontsize=13)
axes[1].set_xlabel('Job Satisfaction Level')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()


## 9. PCA on survey items (train fold only)

Imputer, scaler, and PCA are all **fit on the training fold** and then applied to test.
The sentinel −1 is replaced with NaN before PCA so it doesn't pollute the numeric space.


In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

X_pca_train = df_train[SURVEY_COLS].replace(-1, np.nan)
X_pca_test  = df_test[SURVEY_COLS].replace(-1, np.nan)

pca_imputer = SimpleImputer(strategy='median')
X_tr_imp = pca_imputer.fit_transform(X_pca_train)   # fit on train
X_te_imp = pca_imputer.transform(X_pca_test)         # apply to test

scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_tr_imp)             # fit on train
X_te_sc = scaler.transform(X_te_imp)                 # apply to test

pca = PCA()
pca.fit(X_tr_sc)                                     # fit on train

cumvar = np.cumsum(pca.explained_variance_ratio_ * 100)
n_90   = int(np.argmax(cumvar >= 90)) + 1
print(f"Components to retain 90% variance: {n_90} / {len(SURVEY_COLS)}")

plt.figure(figsize=(8, 4))
plt.plot(cumvar, marker='o', markersize=3, color='steelblue')
plt.axhline(90, color='red',    linestyle='--', label='90% threshold')
plt.axvline(n_90 - 1, color='orange', linestyle='--', label=f'{n_90} components')
plt.xlabel('Number of components')
plt.ylabel('Cumulative explained variance (%)')
plt.title('PCA — Survey Features (fit on train)')
plt.legend()
plt.tight_layout()
plt.show()


## 10. Random Forest — feature importance

Encoding and imputation are fit on the training fold and applied to the test fold.
Nulls in `CompTotal_USD` are median-imputed (train median only).
No rows are dropped during modelling.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report

FINAL_FEATURES = ['CompTotal_USD', 'Country', 'TechEndorseIntro'] + SURVEY_COLS
CAT_COLS = ['Country', 'TechEndorseIntro']

X_train = df_train[FINAL_FEATURES].copy()
X_test  = df_test[FINAL_FEATURES].copy()
y_train = df_train[TARGET]
y_test  = df_test[TARGET]

# Encode categoricals — fit on train only
enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-2)
X_train[CAT_COLS] = enc.fit_transform(X_train[CAT_COLS].astype(str))
X_test[CAT_COLS]  = enc.transform(X_test[CAT_COLS].astype(str))

# Impute remaining numerics — fit on train only
imp = SimpleImputer(strategy='median')
X_train_imp = pd.DataFrame(imp.fit_transform(X_train), columns=FINAL_FEATURES)
X_test_imp  = pd.DataFrame(imp.transform(X_test),      columns=FINAL_FEATURES)

print("Training Random Forest …")
rf = RandomForestClassifier(
    n_estimators=200, max_depth=12, random_state=42,
    n_jobs=-1, class_weight='balanced'
)
rf.fit(X_train_imp, y_train)
preds = rf.predict(X_test_imp)

print("\n--- MODEL PERFORMANCE ---")
print(classification_report(y_test, preds))

imp_df = (
    pd.DataFrame({'Feature': FINAL_FEATURES, 'Importance': rf.feature_importances_})
    .sort_values('Importance')
)
plt.figure(figsize=(10, 7))
top = imp_df.tail(15)
plt.barh(top['Feature'], top['Importance'], color='teal')
plt.xlabel('Relative Importance (Gini)')
plt.title('Top 15 Drivers of Job Satisfaction — Random Forest')
plt.tight_layout()
plt.show()


## 11. Champion (18 raw survey cols) vs Challenger (4 aggregated themes)

Aggregated theme scores are computed **per fold separately** after the split.
The sentinel −1 is excluded from means by replacing with NaN locally.


In [ ]:
from sklearn.metrics import confusion_matrix

MACRO = ['CompTotal_USD', 'Country']

def build_agg_features(df_fold):
    """Compute aggregated theme features. Sentinel −1 excluded from means."""    tmp = df_fold[SURVEY_COLS].replace(-1, np.nan)
    agg = pd.DataFrame(index=df_fold.index)
    agg['Endorse_Reputation'] = tmp[['TechEndorse_5', 'TechEndorse_6', 'TechEndorse_7']].mean(axis=1)
    agg['Endorse_Pragmatic']  = tmp[['TechEndorse_8', 'TechEndorse_9']].mean(axis=1)
    agg['Oppose_Friction']    = tmp[['TechOppose_3',  'TechOppose_5']].mean(axis=1)
    agg['Skipped_Complaints'] = (df_fold['TechOppose_15'] == -1).astype(int)
    return agg.fillna(0)

AGG_COLS = ['Endorse_Reputation', 'Endorse_Pragmatic', 'Oppose_Friction', 'Skipped_Complaints']

agg_train = build_agg_features(df_train)
agg_test  = build_agg_features(df_test)

X_champ_train = pd.concat([df_train[MACRO + SURVEY_COLS].reset_index(drop=True)], axis=1)
X_champ_test  = pd.concat([df_test[MACRO  + SURVEY_COLS].reset_index(drop=True)], axis=1)
X_chall_train = pd.concat([df_train[MACRO].reset_index(drop=True), agg_train.reset_index(drop=True)], axis=1)
X_chall_test  = pd.concat([df_test[MACRO].reset_index(drop=True),  agg_test.reset_index(drop=True)],  axis=1)

y_tr = df_train[TARGET].reset_index(drop=True)
y_te = df_test[TARGET].reset_index(drop=True)

for X_tr, X_te in [(X_champ_train, X_champ_test), (X_chall_train, X_chall_test)]:
    enc_c = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-2)
    X_tr[['Country']] = enc_c.fit_transform(df_train[['Country']].astype(str))
    X_te[['Country']] = enc_c.transform(df_test[['Country']].astype(str))

for X_tr, X_te in [(X_champ_train, X_champ_test), (X_chall_train, X_chall_test)]:
    imp_c = SimpleImputer(strategy='median')
    cols = X_tr.columns.tolist()
    X_tr[:] = imp_c.fit_transform(X_tr)
    X_te[:] = imp_c.transform(X_te)

rf_champ = RandomForestClassifier(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1, class_weight='balanced')
rf_chall = RandomForestClassifier(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1, class_weight='balanced')
rf_champ.fit(X_champ_train, y_tr)
rf_chall.fit(X_chall_train, y_tr)

p_champ = rf_champ.predict(X_champ_test)
p_chall = rf_chall.predict(X_chall_test)

print("=== CHAMPION — raw survey cols ===")
print(classification_report(y_te, p_champ))
print("=== CHALLENGER — 4 aggregated themes ===")
print(classification_report(y_te, p_chall))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
classes = rf_champ.classes_
sns.heatmap(confusion_matrix(y_te, p_champ), annot=True, fmt='d', cmap='Blues',
            ax=axes[0], xticklabels=classes, yticklabels=classes)
axes[0].set_title('Champion — raw survey cols')
sns.heatmap(confusion_matrix(y_te, p_chall), annot=True, fmt='d', cmap='Oranges',
            ax=axes[1], xticklabels=classes, yticklabels=classes)
axes[1].set_title('Challenger — 4 aggregated themes')
for ax in axes:
    ax.set_ylabel('Actual'); ax.set_xlabel('Predicted')
plt.tight_layout()
plt.show()


## 12. Currency relevance — raw feature vs `is_foreign_currency`

**Research question:** Is the high-cardinality `Currency` string a better predictor
of `JobSatLevel` than a binary `is_foreign_currency` flag?

**Flag definition:** `is_foreign_currency = 1` when the respondent's reported currency
differs from the **dominant currency of their country** (computed as the mode in the
training fold). This is more precise than a USD-only comparison — a developer in Japan
reporting in EUR is also flagged.

The dominant-currency map is derived from the training fold only and applied to test,
so there is no leakage.

Both features are categorical → target, so **Cramér's V** is the correct metric here.


In [ ]:
# Engineer binary flag using train-derived dominant currency per country
dominant_currency_train = (
    df_train.dropna(subset=['Country', 'Currency'])
    .groupby('Country')['Currency']
    .agg(lambda x: x.mode()[0])
    .to_dict()
)

def is_foreign(row):
    dominant = dominant_currency_train.get(row['Country'], None)
    if dominant is None or pd.isna(row['Currency']):
        return 0  # unknown country — conservative default
    return int(row['Currency'] != dominant)

df_train['is_foreign_currency'] = df_train[['Country', 'Currency']].apply(is_foreign, axis=1)
# Apply the SAME train-derived map to test — no test statistics used
df_test['is_foreign_currency']  = df_test[['Country', 'Currency']].apply(is_foreign, axis=1)

print(f"Train — is_foreign_currency=1: {df_train['is_foreign_currency'].sum():,}  ({df_train['is_foreign_currency'].mean()*100:.1f}%)")
print(f"Test  — is_foreign_currency=1: {df_test['is_foreign_currency'].sum():,}  ({df_test['is_foreign_currency'].mean()*100:.1f}%)")

# --- Country → Currency redundancy check (train only) ---
cur_check = df_train.groupby('Country')['Currency'].nunique()
total_c   = len(cur_check)
single_c  = (cur_check == 1).sum()
redundancy_pct = single_c / total_c * 100

print(f"\nUnique countries (train)                : {total_c}")
print(f"Countries with exactly 1 currency       : {single_c}  ({redundancy_pct:.1f}%)")

# --- Cramér's V comparison ---
def cramers_v(x, y):
    ct   = pd.crosstab(x, y)
    chi2 = ss.chi2_contingency(ct)[0]
    n    = ct.sum().sum()
    r, k = ct.shape
    phi2corr = max(0, chi2 / n - ((k - 1) * (r - 1)) / (n - 1))
    rcorr = r - ((r - 1) ** 2) / (n - 1)
    kcorr = k - ((k - 1) ** 2) / (n - 1)
    m = min(kcorr - 1, rcorr - 1)
    return 0.0 if m <= 0 else np.sqrt(phi2corr / m)

temp_cur  = df_train[['Currency', TARGET]].dropna()
temp_flag = df_train[['is_foreign_currency', TARGET]].dropna()

cv_raw  = cramers_v(temp_cur['Currency'].astype(str), temp_cur[TARGET])
cv_flag = cramers_v(temp_flag['is_foreign_currency'].astype(str), temp_flag[TARGET])

_, p_raw,  _, _ = ss.chi2_contingency(pd.crosstab(temp_cur['Currency'].astype(str), temp_cur[TARGET]))
_, p_flag, _, _ = ss.chi2_contingency(pd.crosstab(temp_flag['is_foreign_currency'].astype(str), temp_flag[TARGET]))

print(f"\n{'Feature':<25} {'Cramér V':>10}  {'p-value':>12}")
print(f"{'Currency (raw)':<25} {cv_raw:>10.4f}  {p_raw:>12.4e}")
print(f"{'is_foreign_currency':<25} {cv_flag:>10.4f}  {p_flag:>12.4e}")

print("\n--- Verdict ---")
delta = cv_raw - cv_flag
if delta > 0.01:
    print(f"Raw Currency is stronger by ΔV = {delta:.4f}.")
    print(f"However, with {redundancy_pct:.0f}% Country→Currency overlap, raw Currency")
    print("adds cardinality without meaningful new signal. Prefer 'Country' directly.")
elif delta < -0.01:
    print(f"is_foreign_currency is stronger by ΔV = {abs(delta):.4f}.")
    print("The binary flag is a BETTER feature: higher signal, far lower cardinality.")
    print("Recommendation: replace raw Currency with is_foreign_currency.")
else:
    print(f"Both features have nearly identical association (ΔV = {abs(delta):.4f}).")
    print("Prefer is_foreign_currency for parsimony — same signal, 1 dimension vs many.")

# --- Visual ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

flag_pct = (
    df_train.groupby([TARGET, 'is_foreign_currency'])
    .size().unstack(fill_value=0)
    .apply(lambda r: r / r.sum() * 100, axis=1)
)
flag_pct.plot(kind='bar', ax=axes[0], colormap='Set2', edgecolor='black')
axes[0].set_title("is_foreign_currency distribution by JobSatLevel (train)", fontsize=12)
axes[0].set_xlabel('JobSatLevel')
axes[0].set_ylabel('% of respondents')
axes[0].tick_params(axis='x', rotation=30)
axes[0].legend(["Same as country's main (0)", "Foreign currency (1)"])

bars = axes[1].bar(
    ["Currency\n(raw)", "is_foreign\n_currency"],
    [cv_raw, cv_flag],
    color=['steelblue', 'coral'], edgecolor='black', width=0.4
)
axes[1].set_ylim(0, max(cv_raw, cv_flag) * 1.5)
axes[1].set_ylabel("Cramér's V")
axes[1].set_title("Predictive association with JobSatLevel\n(Cramér's V, train fold)", fontsize=12)
for bar, val in zip(bars, [cv_raw, cv_flag]):
    axes[1].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 0.002,
                 f'{val:.4f}', ha='center', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()
